# Context reranker v2 Colab workflow

Colab is the GPU training furnace; local Windows/WSL is the audit, smoke, and acceptance station. This notebook uses `training/context_reranker_v2.py` only, copies the new corpus into `/content/golf-ime-data/context_v2/`, runs audit first, runs a tiny smoke/sanity pass, and then optionally runs the full training job into Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone


def run(cmd, *, cwd=None, stdout_path=None, stderr_path=None, env=None):
    cmd = [str(part) for part in cmd]
    print('$ ' + ' '.join(cmd), flush=True)
    stdout_file = open(stdout_path, 'w', encoding='utf-8') if stdout_path else None
    stderr_file = open(stderr_path, 'w', encoding='utf-8') if stderr_path else None
    try:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            text=True,
            stdout=stdout_file,
            stderr=stderr_file if stderr_file else subprocess.STDOUT if stdout_file else None,
        )
    finally:
        if stdout_file:
            stdout_file.close()
        if stderr_file:
            stderr_file.close()
    if proc.returncode != 0:
        for path in (stderr_path, stdout_path):
            if path and Path(path).is_file():
                print(Path(path).read_text(encoding='utf-8')[-4000:])
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')
    return proc

BRANCH = 'prepare-context-reranker-v2-colab'
REPO_URL = 'https://github.com/3516027002att-ui/Golf-Input-Method.git'
REPO_DIR = Path('/content/Golf-Input-Method')

shutil.rmtree(REPO_DIR, ignore_errors=True)
run(['git', 'clone', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
run(['git', 'fetch', 'origin', f'{BRANCH}:{BRANCH}'])
run(['git', 'checkout', BRANCH])
run(['git', 'status', '--short', '--branch'])


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-train.txt'])


In [ ]:
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/golf-ime-data-rebuild/clean_dataset_v3/left_context_only')
LOCAL_DATA_ROOT = Path('/content/golf-ime-data/context_v2')
RUN_DIR = Path('/content/drive/MyDrive/golf-ime-runs/context_reranker_v2_new_corpus')
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

source_files = {
    'train': DRIVE_DATA_ROOT / 'train_new_corpus.jsonl',
    'val': DRIVE_DATA_ROOT / 'val_new_corpus_v2.jsonl',
    'test': DRIVE_DATA_ROOT / 'test_new_corpus_v2.jsonl',
}
legacy_train = DRIVE_DATA_ROOT / 'train.jsonl'
local_files = {}
for split, src in source_files.items():
    if not src.is_file():
        raise FileNotFoundError(src)
    dst = LOCAL_DATA_ROOT / src.name
    shutil.copy2(src, dst)
    local_files[split] = dst
    print(split, dst, dst.stat().st_size)
print('excluded first-round train:', legacy_train)


In [ ]:
AUDIT_REPORT = RUN_DIR / 'context_v2_new_corpus_audit.md'
run([
    sys.executable, 'training/context_reranker_v2.py', 'audit-splits',
    '--train', local_files['train'],
    '--val', local_files['val'],
    '--test', local_files['test'],
    '--report', AUDIT_REPORT,
])
print(AUDIT_REPORT)


In [ ]:
def copy_head(src: Path, dst: Path, limit: int) -> int:
    written = 0
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open('r', encoding='utf-8-sig') as reader, dst.open('w', encoding='utf-8', newline='\n') as writer:
        for line in reader:
            if written >= limit:
                break
            if line.strip():
                writer.write(line.rstrip('\n') + '\n')
                written += 1
    if written == 0:
        raise RuntimeError(f'No rows copied from {src}')
    return written

SMOKE_DIR = Path('/content/golf-ime-data/context_v2_smoke')
SMOKE_TRAIN = SMOKE_DIR / 'train_smoke.jsonl'
SMOKE_VAL = SMOKE_DIR / 'val_smoke.jsonl'
SMOKE_TEST = SMOKE_DIR / 'test_smoke.jsonl'
smoke_rows = {
    'train': copy_head(local_files['train'], SMOKE_TRAIN, 64),
    'val': copy_head(local_files['val'], SMOKE_VAL, 24),
    'test': copy_head(local_files['test'], SMOKE_TEST, 24),
}
print(smoke_rows)


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SMOKE_ENCODER = 'hf-internal-testing/tiny-random-bert'
SMOKE_CHECKPOINT = RUN_DIR / 'smoke_checkpoint'
SMOKE_RANDOM_CHECKPOINT = RUN_DIR / 'smoke_random_label_checkpoint'
print('device=', DEVICE, 'encoder=', SMOKE_ENCODER)


In [ ]:
run([
    sys.executable, 'training/context_reranker_v2.py', 'train',
    '--train', SMOKE_TRAIN,
    '--val', SMOKE_VAL,
    '--output-dir', SMOKE_CHECKPOINT,
    '--encoder', SMOKE_ENCODER,
    '--epochs', '1',
    '--batch-size', '2',
    '--eval-batch-size', '4',
    '--max-length', '64',
    '--context-before-chars', '48',
    '--context-after-chars', '24',
    '--context-mode', 'online',
    '--log-every', '1',
    '--device', DEVICE,
])
print(SMOKE_CHECKPOINT)


In [ ]:
SMOKE_EVAL_JSON = RUN_DIR / 'smoke_eval.json'
SMOKE_EVAL_LOG = RUN_DIR / 'smoke_eval.stderr.log'
SMOKE_NO_CONTEXT_JSON = RUN_DIR / 'smoke_no_context_eval.json'
SMOKE_NO_CONTEXT_LOG = RUN_DIR / 'smoke_no_context_eval.stderr.log'
SMOKE_SHUFFLE_JSON = RUN_DIR / 'smoke_shuffle_eval.json'
SMOKE_SHUFFLE_LOG = RUN_DIR / 'smoke_shuffle_eval.stderr.log'

run([
    sys.executable, 'training/context_reranker_v2.py', 'eval',
    '--data', SMOKE_TEST,
    '--checkpoint', SMOKE_CHECKPOINT,
    '--context-mode', 'online',
    '--device', DEVICE,
], stdout_path=SMOKE_EVAL_JSON, stderr_path=SMOKE_EVAL_LOG)
run([
    sys.executable, 'training/context_reranker_v2.py', 'eval',
    '--data', SMOKE_TEST,
    '--checkpoint', SMOKE_CHECKPOINT,
    '--context-mode', 'none',
    '--device', DEVICE,
], stdout_path=SMOKE_NO_CONTEXT_JSON, stderr_path=SMOKE_NO_CONTEXT_LOG)
run([
    sys.executable, 'training/context_reranker_v2.py', 'eval',
    '--data', SMOKE_TEST,
    '--checkpoint', SMOKE_CHECKPOINT,
    '--context-mode', 'online',
    '--candidate-order', 'shuffle',
    '--device', DEVICE,
], stdout_path=SMOKE_SHUFFLE_JSON, stderr_path=SMOKE_SHUFFLE_LOG)
print(SMOKE_EVAL_JSON)
print(SMOKE_NO_CONTEXT_JSON)
print(SMOKE_SHUFFLE_JSON)


In [ ]:
run([
    sys.executable, 'training/context_reranker_v2.py', 'train',
    '--train', SMOKE_TRAIN,
    '--val', SMOKE_VAL,
    '--output-dir', SMOKE_RANDOM_CHECKPOINT,
    '--encoder', SMOKE_ENCODER,
    '--epochs', '1',
    '--batch-size', '2',
    '--eval-batch-size', '4',
    '--max-length', '64',
    '--context-before-chars', '48',
    '--context-after-chars', '24',
    '--context-mode', 'online',
    '--label-mode', 'random',
    '--log-every', '1',
    '--device', DEVICE,
])

SMOKE_RANDOM_EVAL_JSON = RUN_DIR / 'smoke_random_label_eval.json'
SMOKE_RANDOM_EVAL_LOG = RUN_DIR / 'smoke_random_label_eval.stderr.log'
run([
    sys.executable, 'training/context_reranker_v2.py', 'eval',
    '--data', SMOKE_TEST,
    '--checkpoint', SMOKE_RANDOM_CHECKPOINT,
    '--context-mode', 'online',
    '--device', DEVICE,
], stdout_path=SMOKE_RANDOM_EVAL_JSON, stderr_path=SMOKE_RANDOM_EVAL_LOG)
print(SMOKE_RANDOM_EVAL_JSON)


In [ ]:
def read_eval_json(path: Path) -> dict:
    text = path.read_text(encoding='utf-8')
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end < start:
        raise ValueError(f'No JSON object in {path}')
    return json.loads(text[start:end + 1])

smoke_eval = read_eval_json(SMOKE_EVAL_JSON)
smoke_no_context = read_eval_json(SMOKE_NO_CONTEXT_JSON)
smoke_shuffle = read_eval_json(SMOKE_SHUFFLE_JSON)
smoke_random = read_eval_json(SMOKE_RANDOM_EVAL_JSON)
smoke_summary = {
    'online_top1': smoke_eval['metrics']['top1'],
    'no_context_top1': smoke_no_context['metrics']['top1'],
    'shuffle_top1': smoke_shuffle['metrics']['top1'],
    'random_label_top1': smoke_random['metrics']['top1'],
    'random_baseline_top1': smoke_random['random_baseline']['top1'],
    'online_red_lines': smoke_eval.get('red_line_findings', []),
    'random_label_red_lines': smoke_random.get('red_line_findings', []),
}
SMOKE_SUMMARY_JSON = RUN_DIR / 'smoke_summary.json'
SMOKE_SUMMARY_JSON.write_text(json.dumps(smoke_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(smoke_summary, ensure_ascii=False, indent=2))
if smoke_summary['online_red_lines'] or smoke_summary['random_label_red_lines']:
    print('Smoke eval red lines were reported; inspect before full training.')


In [ ]:
PREDICT_JSON = RUN_DIR / 'smoke_predict_xuexiao.json'
PREDICT_CONTEXT = "\u6211\u4eca\u5929\u60f3\u53bb"
PREDICT_CANDIDATES_JSON = json.dumps(["\u5b66\u6821", "\u7761\u89c9", "\u5317\u4eac", "\u4e00\u4e2a"], ensure_ascii=False)
run([
    sys.executable, 'scripts/predict_context_reranker_v2.py',
    '--checkpoint', SMOKE_CHECKPOINT,
    '--context-before', PREDICT_CONTEXT,
    '--composing', 'xuexiao',
    '--candidates-json', PREDICT_CANDIDATES_JSON,
    '--device', DEVICE,
], stdout_path=PREDICT_JSON)
print(PREDICT_JSON.read_text(encoding='utf-8'))


In [ ]:
RUN_FULL_TRAIN = False  # Flip to True only after audit and smoke/sanity look sane.
FULL_ENCODER = 'hfl/chinese-macbert-base'
FULL_EPOCHS = 3
FULL_BATCH_SIZE = 4
FULL_EVAL_BATCH_SIZE = 8
FULL_CHECKPOINT = RUN_DIR / 'checkpoint'
FULL_TRAIN_LOG = RUN_DIR / 'full_train.log'

if RUN_FULL_TRAIN:
    run([
        sys.executable, 'training/context_reranker_v2.py', 'train',
        '--train', local_files['train'],
        '--val', local_files['val'],
        '--output-dir', FULL_CHECKPOINT,
        '--encoder', FULL_ENCODER,
        '--epochs', str(FULL_EPOCHS),
        '--batch-size', str(FULL_BATCH_SIZE),
        '--eval-batch-size', str(FULL_EVAL_BATCH_SIZE),
        '--context-mode', 'online',
        '--device', DEVICE,
        '--log-every', '50',
    ], stdout_path=FULL_TRAIN_LOG)
    print('full checkpoint:', FULL_CHECKPOINT)
else:
    print('RUN_FULL_TRAIN is False. Set it to True for the long GPU training run.')


In [ ]:
FULL_EVAL_JSON = RUN_DIR / 'full_eval.json'
FULL_TEST_EVAL_JSON = RUN_DIR / 'full_test_eval.json'
FULL_EVAL_LOG = RUN_DIR / 'full_eval.stderr.log'
FULL_TEST_EVAL_LOG = RUN_DIR / 'full_test_eval.stderr.log'
MANIFEST_JSON = RUN_DIR / 'run_manifest.json'

if RUN_FULL_TRAIN:
    run([
        sys.executable, 'training/context_reranker_v2.py', 'eval',
        '--data', local_files['val'],
        '--checkpoint', FULL_CHECKPOINT,
        '--batch-size', str(FULL_EVAL_BATCH_SIZE),
        '--context-mode', 'online',
        '--device', DEVICE,
    ], stdout_path=FULL_EVAL_JSON, stderr_path=FULL_EVAL_LOG)
    read_eval_json(FULL_EVAL_JSON)
    run([
        sys.executable, 'training/context_reranker_v2.py', 'eval',
        '--data', local_files['test'],
        '--checkpoint', FULL_CHECKPOINT,
        '--batch-size', str(FULL_EVAL_BATCH_SIZE),
        '--context-mode', 'online',
        '--device', DEVICE,
    ], stdout_path=FULL_TEST_EVAL_JSON, stderr_path=FULL_TEST_EVAL_LOG)
    read_eval_json(FULL_TEST_EVAL_JSON)

manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'branch': BRANCH,
    'repo_url': REPO_URL,
    'run_dir': str(RUN_DIR),
    'data_root': str(DRIVE_DATA_ROOT),
    'local_data_root': str(LOCAL_DATA_ROOT),
    'source_files': {key: str(value) for key, value in source_files.items()},
    'excluded_first_round_train': str(legacy_train),
    'audit_report': str(AUDIT_REPORT),
    'smoke': {
        'rows': smoke_rows,
        'checkpoint': str(SMOKE_CHECKPOINT),
        'summary_json': str(SMOKE_SUMMARY_JSON),
        'predict_json': str(PREDICT_JSON),
    },
    'full_train': {
        'ran': bool(RUN_FULL_TRAIN),
        'encoder': FULL_ENCODER,
        'checkpoint': str(FULL_CHECKPOINT),
        'train_log': str(FULL_TRAIN_LOG),
        'val_eval_json': str(FULL_EVAL_JSON),
        'val_eval_stderr_log': str(FULL_EVAL_LOG),
        'test_eval_json': str(FULL_TEST_EVAL_JSON),
        'test_eval_stderr_log': str(FULL_TEST_EVAL_LOG),
    },
}
MANIFEST_JSON.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(MANIFEST_JSON)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
